# 01 — Problem Understanding

## Prediction Problem
**Predict, at the moment of patient discharge, whether a diabetic patient 
will be readmitted within 30 days.**

## Target Definition
Binary classification:
- 1 = readmitted within 30 days (`readmitted == '<30'`)
- 0 = not readmitted within 30 days (`readmitted` is `'>30'` or `'NO'`)

Rationale: 30-day readmissions carry financial penalties under US healthcare 
policy. This is the actionable business target.

## Prediction Timing (Leakage Boundary)
The prediction is made **at discharge**. Only features knowable at or before 
discharge are valid. Any feature describing post-discharge events is leakage 
and must be excluded.

## Grain
Each row = one hospital encounter (visit), identified by `encounter_id`.
NOTE: Patients (`patient_nbr`) can appear in multiple rows. This must be 
handled in the train/test split to avoid the same patient appearing in 
both sets (a form of leakage).

## Class Balance
~11% of encounters are <30 readmissions. Imbalanced classification — 
accuracy will be a misleading metric; will use precision, recall, ROC-AUC.

## Business Use
A risk score at discharge lets care teams target follow-up resources 
(calls, home visits, medication checks) at high-risk patients before 
they leave.

In [1]:
import pandas as pd

df = pd.read_csv('../data/diabetic_data.csv')

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df['readmitted'].value_counts())
print("\nColumns:")
print(df.columns.tolist())

Shape: (101766, 50)

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


## Leakage Audit

### Target (predicted, not a feature)
| readmitted | target | binarized: '<30' → 1, else → 0 |

### Identifiers (excluded from features)
| encounter_id | identifier | unique per encounter |
| patient_nbr | identifier | unique per patient — used for grouped train/test split |

### Requires row filtering (kept as feature, but some rows dropped)
| discharge_disposition_id | legal* | known at discharge; drop expired/hospice encounters (can't be readmitted) |

### Legal features
| feature | legal/leakage/identifier | one-line reason |
| ------- | ------------------------ | --------------- |
| race | legal | known at admision |
| gender | legal | known at admision |
| age | legal | known at admision |
| weight | legal | known at admision |
| admission_type_id | legal | known during admission |
| admission_source_id | legal | known at admission |
| time_in_hospital | legal | known by discharge |
| num_lab_procedures | legal | known by discharge |
| num_medications | legal | known by discharge |
| payer_code | legal | known at admission |
| medical_specialty | legal | known at admission - the admitting physician's specialty  |
| num_procedures | legal | known by discharge |
| number_outpatient | legal | historical data known at admission |
| number_emergency | legal | historical data known at admission |
| number_inpatient | legal | historical data known at admission |
| diag_1 | legal | known during stay |
| diag_2 | legal | known during stay |
| diag_3 | legal | known during stay |
| number_diagnoses | legal | known during stay |
| max_glu_serum | legal | known during stay |
| A1Cresult | legal | known during stay |
| metformin | legal | known during stay |
| repaglinide | legal | known during stay |
| nateglinide | legal| known during stay |
| chlorpropamide | legal | known during stay |
| discharge_disposition_id | legal | known at discharge; filter out expired/hospice rows in cleaning |
| glimepiride | legal | known during stay |
| acetohexamide | legal | known during stay |
| glipizide | legal | known during stay |
| glyburide | legal | known during stay |
| tolbutamide | legal | known during stay |
| pioglitazone | legal| known during stay |
| rosiglitazone | legal | known during stay |
| acarbose | legal | known during stay |
| miglitol | legal | known during stay |
| troglitazone | legal | known during stay |
| tolazamide | legal | known during stay |
| citoglipton | legal | known during stay |
| glyburide-metformin | legal | known during stay |
| glipizide-metformin | legal | known during stay |
| glimepiride-pioglitazone | legal | known during stay |
| metformin-rosiglitazone | legal | known during stay |
| metformin-pioglitazone | legal | known during stay |
| change | legal | known at discharge |
| diabetesMed | legal | known at discharge |
